# Score `real_obd_001` against the VED-HEV model

Separate notebook from `score_real_vehicle_data.ipynb` (the KIT one) —
**not** a drop-in swap of the model path, for two real reasons:

1. **Different feature set.** The KIT model used `coolant_temp_c_dev`;
   VED-HEV was trained on `engine_load_pct_dev` instead (VED has no
   coolant column at all — see the VED/KIT trade-off discussion in
   `model_details.md`/chat history). Wrong columns entirely, not a rename.
2. **The rolling-deviation computation must match training exactly**
   (train/score symmetry). VED's features were computed with a
   **time-based** 60-second trailing rolling median
   (`ved_feature_engineering.py`), not a row-based one — this notebook
   reproduces that exact logic, not KIT's.

This session (`real_obd_001`) has no VED-style `Trip` boundary — the
whole session is treated as one continuous trip, same as `OBD2Lib`/the
Pi stack already treats it (a session is just a named time window, per
`architecture.md` §3.3).

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from db import query  # analytics/src/db.py — DISPLAY_TZ-converting query()

## Pull `real_obd_001` via the time-range join

Same pattern as everywhere else in this project post-§3.3 migration
(`schema-reference.md`) — `sessions.id` join on a time range, not an
equality filter on a `telemetry.session_id` column (that column doesn't
exist anymore).

In [2]:
SESSION_ID = "real_obd_001"

sql = """
    SELECT t.time, t.rpm, t.engine_load_pct, t.speed_kmh
    FROM telemetry t
    JOIN sessions s
      ON t.time BETWEEN s.started_at AND COALESCE(s.ended_at, NOW())
    WHERE s.id = %(session_id)s
    ORDER BY t.time
"""
raw = query(sql, params={"session_id": SESSION_ID})
print(f"{len(raw):,} rows, {raw['time'].min()} to {raw['time'].max()}")
raw.head()

2,410 rows, 2026-07-26 15:17:21.037710+05:30 to 2026-07-26 15:49:10.696250+05:30


,time,rpm,engine_load_pct,speed_kmh
0,2026-07-26 15:17:21.037710+05:30,0.0,0.0,0.0
1,2026-07-26 15:17:21.933809+05:30,0.0,0.0,0.0
2,2026-07-26 15:17:22.723591+05:30,0.0,0.0,0.0
3,2026-07-26 15:17:23.212319+05:30,0.0,0.0,0.0
4,2026-07-26 15:17:23.739199+05:30,0.0,0.0,0.0


## Feature engineering — must mirror `ved_feature_engineering.py` exactly

Same `ROLLING_WINDOW`/`MIN_TRIP_SECONDS` constants, same trailing
time-based rolling median, same cold-start trim. The whole session is
one continuous "trip" (no `groupby` needed — that's the one structural
difference from the VED pipeline, which groups by `(VehId, Trip)`;
here there's only ever one vehicle, one session).

In [3]:
ROLLING_WINDOW = "60s"      # must match ved_feature_engineering.py
MIN_TRIP_SECONDS = 60       # must match ved_feature_engineering.py

df = raw.sort_values("time").set_index("time")

df["rpm_dev"] = df["rpm"] - df["rpm"].rolling(ROLLING_WINDOW).median()
df["engine_load_pct_dev"] = df["engine_load_pct"] - df["engine_load_pct"].rolling(ROLLING_WINDOW).median()

df = df.reset_index()
cutoff = df["time"].iloc[0] + pd.Timedelta(seconds=MIN_TRIP_SECONDS)
df = df[df["time"] >= cutoff].copy()

print(f"{len(df):,} rows after cold-start trim")
df[["rpm_dev", "engine_load_pct_dev", "speed_kmh"]].describe()

2,323 rows after cold-start trim


,rpm_dev,engine_load_pct_dev,speed_kmh
count,2313.000000,2301.000000,2317.000000
mean,203.688824,9.127149,28.344842
std,775.162310,45.547683,20.067841
min,-1703.375000,-94.510000,0.000000
25%,0.000000,0.000000,9.000000
50%,0.000000,0.000000,29.000000
75%,154.250000,2.353000,45.000000
max,2312.500000,99.216000,69.000000


## Score against the VED-HEV model

Fixed threshold locked in during calibration (`ved_threshold_calibration.py`,
3% target FPR row) — **not** recomputed from this session's own scores.
Update `THRESHOLD` below if a different row was ultimately chosen.

In [4]:
MODEL_PATH = Path("../models/ved_hev_isolation_forest.joblib")
THRESHOLD = -0.091278  # locked in from val calibration — see chat/threshold_calibration output

bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
feature_cols = bundle["feature_cols"]
assert feature_cols == ["rpm_dev", "engine_load_pct_dev", "speed_kmh"], feature_cols

scoreable = df.dropna(subset=feature_cols).copy()
scoreable["score"] = model.decision_function(scoreable[feature_cols].to_numpy())
scoreable["anomalous"] = scoreable["score"] < THRESHOLD

n_dropped = len(df) - len(scoreable)
print(f"{n_dropped:,} of {len(df):,} rows dropped (null features), {len(scoreable):,} scored")
print(f"\nOverall anomaly rate: {100*scoreable['anomalous'].mean():.1f}%  "
      f"(expected ~3% if this session looks like ordinary HEV driving)")

34 of 2,323 rows dropped (null features), 2,289 scored

Overall anomaly rate: 13.8%  (expected ~3% if this session looks like ordinary HEV driving)


## Breakdown, same diagnostic as `model_details.md` §7.1

Reproducing the original KIT-model investigation's three checks
(stationary vs. moving, engine-on vs. EV-only, and continuous
engine-on stretch length) against the *new* model — this is the direct
comparison point for whether the VED-HEV model actually resolves the
short-engine-cycle limitation that model was built to address.

In [5]:
stationary = scoreable[scoreable["speed_kmh"] == 0]
moving = scoreable[scoreable["speed_kmh"] > 0]
print(f"Stationary anomaly rate: {100*stationary['anomalous'].mean():.1f}%  (n={len(stationary):,})")
print(f"Moving anomaly rate:     {100*moving['anomalous'].mean():.1f}%  (n={len(moving):,})")

engine_off = scoreable[scoreable["rpm"] == 0]
engine_on = scoreable[scoreable["rpm"] > 0]
print(f"\nEngine-off (EV-mode) anomaly rate: {100*engine_off['anomalous'].mean():.1f}%  (n={len(engine_off):,})")
print(f"Engine-on anomaly rate:            {100*engine_on['anomalous'].mean():.1f}%  (n={len(engine_on):,})")

Stationary anomaly rate: 13.2%  (n=364)
Moving anomaly rate:     13.9%  (n=1,925)

Engine-off (EV-mode) anomaly rate: 5.7%  (n=1,532)
Engine-on anomaly rate:            30.0%  (n=757)


In [6]:
# Continuous engine-on stretch length — the specific mechanism that
# caused the KIT model's 16.7% false-elevated rate (model_details.md §7.1):
# every engine-on row occurred within 60s of the engine restarting, so the
# 60s rolling window never stabilized. Checking whether that's still true
# against real driving (independent of which model is scoring it).
engine_state = (scoreable["rpm"] > 0).astype(int)
stretch_id = (engine_state != engine_state.shift()).cumsum()
on_stretches = scoreable.assign(stretch_id=stretch_id)[engine_state == 1].groupby("stretch_id")["time"]
stretch_lengths = on_stretches.apply(lambda s: (s.max() - s.min()).total_seconds())

print(f"{len(stretch_lengths)} continuous engine-on stretches")
print(stretch_lengths.describe())
print(f"\nStretches >= 60s: {(stretch_lengths >= 60).sum()} of {len(stretch_lengths)}")

56 continuous engine-on stretches
count    56.000000
mean      8.540946
std       8.585695
min       1.487470
25%       2.914703
50%       7.506873
75%      10.049169
max      55.528143
Name: time, dtype: float64

Stretches >= 60s: 0 of 56


## Magnitude check: is `real_obd_001`'s deviation *size* unusual, not just its duration?

The duration-based hypothesis (short engine cycles are underrepresented
in training) was directly checked and **rejected** — VED-HEV's own
vehicles have plenty of short (<60s) engine-on stretches too (median
10.0s, 49.6% under 10s, across ~1.99M rows). So if `real_obd_001` is
still elevated, it's more likely the *magnitude* of the deviation during
those short stretches, not their duration, that differs from what the
model considers typical.

Reference distribution, computed directly from VED-HEV's own short-stretch
(<60s) engine-on rows (train+val+test combined, ~1.99M rows):

| | `rpm_dev` |
|---|---|
| median (abs) | ~480 |
| 90th pct (abs) | ~1,664 |
| 95th pct (abs) | ~1,975 |

Compare `real_obd_001`'s own numbers below against that reference —
values well beyond VED's 90th/95th percentile would point to a genuine
behavioral or measurement difference in this specific vehicle/session,
rather than a training-data gap.

In [7]:
engine_on_rows = scoreable[scoreable["rpm"] > 0]
print("real_obd_001 engine-on rpm_dev (abs value) percentiles:")
print(engine_on_rows["rpm_dev"].abs().describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nreal_obd_001 engine-on engine_load_pct_dev (abs value) percentiles:")
print(engine_on_rows["engine_load_pct_dev"].abs().describe(percentiles=[.5, .75, .9, .95, .99]))

# Fraction of engine-on rows whose |rpm_dev| exceeds VED-HEV's own 95th
# percentile reference (~1975) — a quick "how far outside VED's own
# normal range is this vehicle running" check.
VED_HEV_RPM_DEV_P95 = 1975
frac_beyond_p95 = (engine_on_rows["rpm_dev"].abs() > VED_HEV_RPM_DEV_P95).mean()
print(f"\nFraction of real_obd_001 engine-on rows exceeding VED-HEV's own "
      f"95th-percentile |rpm_dev| ({VED_HEV_RPM_DEV_P95}): {100*frac_beyond_p95:.1f}%")

real_obd_001 engine-on rpm_dev (abs value) percentiles:
count     757.000000
mean     1056.625495
std       652.617301
min         0.000000
50%      1237.500000
75%      1603.750000
90%      1823.550000
95%      1912.000000
99%      2132.830000
max      2312.500000
Name: rpm_dev, dtype: float64

real_obd_001 engine-on engine_load_pct_dev (abs value) percentiles:
count    757.000000
mean      54.240188
std       39.276293
min        0.000000
50%       70.588000
75%       91.373000
90%       93.333000
95%       95.294000
99%       98.039000
max       99.216000
Name: engine_load_pct_dev, dtype: float64

Fraction of real_obd_001 engine-on rows exceeding VED-HEV's own 95th-percentile |rpm_dev| (1975): 3.8%


## Window-duration comparison: does a shorter rolling window help?

`real_obd_001`'s engine-on rows were confirmed to be systematically more
extreme than VED-HEV's own reference distribution — most dramatically for
`engine_load_pct_dev` (median |dev| ~70.6 vs. VED's own reference of
~14.5 at 60s window). Checked directly on VED's own data: shrinking the
window to 30s only shifts VED's own reference distribution modestly
(median |engine_load_pct_dev| 14.5 → 11.8, ~19% — nowhere near closing a
~5x gap), so a window change isn't expected to be the fix on its own.
Testing anyway on `real_obd_001` itself, since a session-specific dynamic
here could behave differently from VED's aggregate shift.

This recomputes features with a 30-second window (mirroring
`ved_feature_engineering.py --window-seconds 30`) and scores against
the separately-trained `ved_hev_w30_isolation_forest.joblib` — a full,
independently trained/calibrated model on the same train/val/test
vehicle split as the 60s model, not just a different scoring threshold
on the same model.

In [8]:
WINDOW_W30 = "30s"
MIN_TRIP_SECONDS_W30 = 30

df_w30 = raw.sort_values("time").set_index("time")
df_w30["rpm_dev"] = df_w30["rpm"] - df_w30["rpm"].rolling(WINDOW_W30).median()
df_w30["engine_load_pct_dev"] = df_w30["engine_load_pct"] - df_w30["engine_load_pct"].rolling(WINDOW_W30).median()
df_w30 = df_w30.reset_index()
cutoff_w30 = df_w30["time"].iloc[0] + pd.Timedelta(seconds=MIN_TRIP_SECONDS_W30)
df_w30 = df_w30[df_w30["time"] >= cutoff_w30].copy()

MODEL_PATH_W30 = Path("../models/ved_hev_w30_isolation_forest.joblib")
# Calibrated (3% target FPR row, val), sealed-test-confirmed: actual test
# FPR 3.698%, load_spike 99.7%, rpm_decorrelation 100.0% — notably better
# than the 60s model's 86.0% load_spike detection at a similar FPR.
THRESHOLD_W30 = -0.089168

bundle_w30 = joblib.load(MODEL_PATH_W30)
model_w30 = bundle_w30["model"]

scoreable_w30 = df_w30.dropna(subset=feature_cols).copy()
scoreable_w30["score"] = model_w30.decision_function(scoreable_w30[feature_cols].to_numpy())
scoreable_w30["anomalous"] = scoreable_w30["score"] < THRESHOLD_W30

engine_on_w30 = scoreable_w30[scoreable_w30["rpm"] > 0]

print(f"{'':20s} {'60s window':>12s} {'30s window':>12s}")
print(f"{'overall anomaly %':20s} {100*scoreable['anomalous'].mean():>11.1f}% {100*scoreable_w30['anomalous'].mean():>11.1f}%")
print(f"{'engine-on anomaly %':20s} {100*engine_on_rows['anomalous'].mean():>11.1f}% {100*engine_on_w30['anomalous'].mean():>11.1f}%")
print(f"{'median |rpm_dev|':20s} {engine_on_rows['rpm_dev'].abs().median():>12.1f} {engine_on_w30['rpm_dev'].abs().median():>12.1f}")
print(f"{'median |load_dev|':20s} {engine_on_rows['engine_load_pct_dev'].abs().median():>12.1f} {engine_on_w30['engine_load_pct_dev'].abs().median():>12.1f}")

                       60s window   30s window
overall anomaly %           13.8%        15.1%
engine-on anomaly %         30.0%        32.0%
median |rpm_dev|           1237.5        875.8
median |load_dev|            70.6         35.5


## Save scored results for the visual comparison notebook

`compare_ved_vs_real_obd_001.ipynb` is a separate notebook process — it
does NOT share this notebook's kernel/variables, even run in sequence.
It loads these saved files instead of assuming any in-memory state.

In [9]:
out_dir = Path("../dataset/processed")
out_dir.mkdir(parents=True, exist_ok=True)
scoreable.to_parquet(out_dir / "real_obd_001_scored_w60.parquet", index=False)
scoreable_w30.to_parquet(out_dir / "real_obd_001_scored_w30.parquet", index=False)
print("Saved real_obd_001_scored_w60.parquet and real_obd_001_scored_w30.parquet")

Saved real_obd_001_scored_w60.parquet and real_obd_001_scored_w30.parquet
